In [ ]:
import os
files = os.listdir("/kaggle/input/single-character-a-images")
print(files)


In [ ]:
%%writefile config.py
#!/usr/bin/env python3
"""
Configuration File for setting up data for other files
"""


# config.py - Configuration classes for brush lettering training
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List
from pathlib import Path
import json

@dataclass
class TrainingConfig:
    """Configuration for LoRA training of brush lettering model"""
    
    # Data settings
    data_dir: str = "./data"
    resolution: int = 512
    
    # Model settings
    model_name: str = "runwayml/stable-diffusion-v1-5"
    
    # LoRA settings
    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: List[str] = field(default_factory=lambda: [
        "to_k", "to_q", "to_v", "to_out.0",
        "proj_in", "proj_out", 
        "ff.net.0.proj", "ff.net.2"
    ])
    
    # Training settings
    num_epochs: int = 50
    batch_size: int = 1
    learning_rate: float = 5e-5
    weight_decay: float = 1e-2
    gradient_accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    
    # Optimizer settings
    use_8bit_adam: bool = False
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    
    # Mixed precision
    mixed_precision: str = "fp16"  # "no", "fp16", "bf16"
    
    # Validation and saving
    validation_steps: int = 100
    save_steps: int = 500
    max_checkpoints: int = 5
    
    # Output settings
    output_dir: str = "./outputs"
    logging_dir: Optional[str] = None
    
    # Noise scheduler settings
    noise_scheduler_type: str = "ddpm"
    prediction_type: str = "epsilon"  # "epsilon" or "v_prediction"
    
    # Data augmentation
    enable_augmentation: bool = True
    augmentation_probability: float = 0.3
    
    # Advanced training settings
    min_snr_gamma: Optional[float] = None  # For Min-SNR weighting
    use_ema: bool = False  # Exponential Moving Average
    ema_decay: float = 0.9999
    
    # Validation generation settings
   # Complete validation_prompts list based on dataset.py character templates
    # This should replace the existing validation_prompts in TrainingConfig

    validation_prompts: List[str] = field(default_factory=lambda: [
    # Lowercase letters (a-z)
    "brush lettering character 'a', elegant calligraphy style",
    "handwritten letter 'b' in brush calligraphy",
    "brush lettering character 'c', artistic style",
    "handwritten letter 'd' in brush calligraphy",
    "elegant brush stroke character 'e'",
    "brush lettering character 'f', calligraphy style",
    "flowing lowercase 'g' with descender loop",
    "elegant lowercase 'h' with tall ascender",
    "simple lowercase 'i' with dot",
    "elegant lowercase 'j' with curved descender",
    "dynamic lowercase 'k' with angled strokes",
    "elegant lowercase 'l' with tall ascender",
    "flowing lowercase 'm' with double arch",
    "elegant lowercase 'n' with smooth arch",
    "circular lowercase 'o' in brush calligraphy",
    "elegant lowercase 'p' with descender",
    "flowing lowercase 'q' with curved tail",
    "elegant lowercase 'r' with smooth shoulder",
    "curved lowercase 's' in brush calligraphy",
    "elegant lowercase 't' with crossbar",
    "flowing lowercase 'u' in brush calligraphy",
    "dynamic lowercase 'v' with angled strokes",
    "wide lowercase 'w' in brush calligraphy",
    "dynamic lowercase 'x' with crossed strokes",
    "flowing lowercase 'y' with descender",
    "dynamic lowercase 'z' with zigzag",
    
    # Uppercase letters (A-Z)
    "brush lettering capital 'A', elegant calligraphy style",
    "elegant uppercase 'B' with double bowls",
    "curved uppercase 'C' in brush calligraphy",
    "elegant uppercase 'D' with curved bowl",
    "structured uppercase 'E' in brush calligraphy",
    "elegant uppercase 'F' with horizontal bars",
    "curved uppercase 'G' in brush calligraphy",
    "structured uppercase 'H' with crossbar",
    "simple uppercase 'I' in brush calligraphy",
    "elegant uppercase 'J' with curved hook",
    "dynamic uppercase 'K' with angled strokes",
    "elegant uppercase 'L' with horizontal base",
    "majestic uppercase 'M' in brush calligraphy",
    "elegant uppercase 'N' with diagonal stroke",
    "circular uppercase 'O' in brush calligraphy",
    "elegant uppercase 'P' with closed bowl",
    "distinctive uppercase 'Q' with tail",
    "elegant uppercase 'R' with bowl and leg",
    "curved uppercase 'S' in brush calligraphy",
    "elegant uppercase 'T' with horizontal top",
    "curved uppercase 'U' in brush calligraphy",
    "dynamic uppercase 'V' with angled strokes",
    "wide uppercase 'W' in brush calligraphy",
    "dynamic uppercase 'X' with crossed strokes",
    "graceful uppercase 'Y' in brush calligraphy",
    "dynamic uppercase 'Z' with zigzag",
    
    # Numbers (0-9)
    "brush lettering number '0', elegant calligraphy style",
    "elegant numeral '1' with clean stem",
    "curved numeral '2' in brush calligraphy",
    "elegant numeral '3' with double curves",
    "angular numeral '4' in brush calligraphy",
    "elegant numeral '5' with curved bottom",
    "curved numeral '6' in brush calligraphy",
    "elegant numeral '7' with angled stroke",
    "curved numeral '8' in brush calligraphy",
    "elegant numeral '9' with curved top",
    
    # Punctuation marks
    "brush lettering period, calligraphy style",
    "curved comma in brush calligraphy",
    "dynamic exclamation point in brush script",
    "curved question mark in brush calligraphy",
    "elegant semicolon in brush script",
    "simple colon in brush calligraphy",
    "elegant dash in brush script",
    "curved apostrophe in brush calligraphy",
    "elegant quotes in brush script",
    
    # Mixed examples for better validation
    "brush lettering word 'art' in flowing script",
    "elegant brush calligraphy word 'ink'",
    "handwritten word 'pen' in brush style",
    "artistic brush lettering 'joy'",
    "flowing brush script word 'zen'",
    "elegant calligraphy word 'mix'",
    "brush lettering phrase 'A1' in artistic style",
    "handwritten characters 'Be' in brush calligraphy",
    "elegant brush stroke 'Go!' with exclamation",
    "artistic brush lettering 'OK?' with question mark"
  ])
    # Memory optimization
    gradient_checkpointing: bool = False
    enable_cpu_offload: bool = False
    
    def __post_init__(self):
        """Post-initialization validation and setup"""
        # Ensure output directory exists
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        
        # Set logging directory if not specified
        if self.logging_dir is None:
            self.logging_dir = str(Path(self.output_dir) / "logs")
        
        # Validate mixed precision setting
        if self.mixed_precision not in ["no", "fp16", "bf16"]:
            raise ValueError(f"Invalid mixed_precision: {self.mixed_precision}")
        
        # Validate model name
        if not self.model_name:
            raise ValueError("model_name cannot be empty")
        
        # Ensure batch size is positive
        if self.batch_size <= 0:
            raise ValueError("batch_size must be positive")
        
        # Ensure learning rate is positive
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive")
    
    @classmethod
    def from_json(cls, json_path: str) -> 'TrainingConfig':
        """Load configuration from JSON file"""
        with open(json_path, 'r') as f:
            config_dict = json.load(f)
        return cls(**config_dict)
    
    def to_json(self, json_path: str):
        """Save configuration to JSON file"""
        # Convert to dict, handling Path objects and other non-serializable types
        config_dict = {}
        for key, value in self.__dict__.items():
            if isinstance(value, Path):
                config_dict[key] = str(value)
            elif isinstance(value, (list, dict, str, int, float, bool)) or value is None:
                config_dict[key] = value
            else:
                config_dict[key] = str(value)
        
        with open(json_path, 'w') as f:
            json.dump(config_dict, f, indent=2)
    
    def get_effective_batch_size(self) -> int:
        """Get the effective batch size considering gradient accumulation"""
        return self.batch_size * self.gradient_accumulation_steps
    
    def get_total_steps(self, dataset_size: int) -> int:
        """Calculate total training steps"""
        steps_per_epoch = dataset_size // self.get_effective_batch_size()
        return steps_per_epoch * self.num_epochs
    
    def update(self, **kwargs):
        """Update configuration with new values"""
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                raise ValueError(f"Invalid configuration key: {key}")

@dataclass 
class DatasetConfig:
    """Configuration for dataset processing"""
    
    # Data paths
    data_dir: str = "./data"
    images_subdir: str = "images"
    prompts_file: Optional[str] = None
    metadata_file: Optional[str] = None
    
    # Data processing
    resolution: int = 512
    train_split: float = 0.9
    random_seed: int = 42
    
    # Image processing
    image_extensions: List[str] = field(default_factory=lambda: [
        '.png', '.jpg', '.jpeg', '.bmp', '.tiff'
    ])
    convert_to_rgb: bool = True
    normalize_range: tuple = (-1, 1)  # Range for SD models
    
    # Augmentation settings
    enable_augmentation: bool = True
    rotation_degrees: float = 5.0
    perspective_distortion: float = 0.1
    brightness_jitter: float = 0.1
    contrast_jitter: float = 0.1
    random_erase_prob: float = 0.1
    
    # Character-specific settings
    supported_characters: List[str] = field(default_factory=lambda: [
        *[chr(i) for i in range(ord('a'), ord('z') + 1)],  # a-z
        *[chr(i) for i in range(ord('A'), ord('Z') + 1)],  # A-Z
        *[chr(i) for i in range(ord('0'), ord('9') + 1)],  # 0-9
        '.', ',', '!', '?', ';', ':', '-', "'", '"'
    ])
    
    def __post_init__(self):
        """Validate dataset configuration"""
        if not 0 < self.train_split < 1:
            raise ValueError("train_split must be between 0 and 1")
        
        if self.resolution <= 0:
            raise ValueError("resolution must be positive")
        
        # Ensure data directory path exists
        if not Path(self.data_dir).exists():
            raise ValueError(f"Data directory does not exist: {self.data_dir}")

@dataclass
class InferenceConfig:
    """Configuration for inference/generation"""
    
    # Model paths
    base_model_path: str = "runwayml/stable-diffusion-v1-5"
    lora_model_path: str = "./outputs/final_model"
    
    # Generation settings
    num_inference_steps: int = 60
    guidance_scale: float = 12
    negative_prompt: str = "blurry, low quality, distorted, ugly, bad anatomy"
    
    # Output settings
    output_resolution: int = 512
    num_images_per_prompt: int = 1
    
    # Sampling settings
    scheduler_type: str = "ddim"  # "ddim", "ddpm", "dpm", "euler"
    eta: float = 0.0
    
    # Safety settings
    safety_checker: bool = False
    nsfw_filter: bool = False
    
    def __post_init__(self):
        """Validate inference configuration"""
        if self.num_inference_steps <= 0:
            raise ValueError("num_inference_steps must be positive")
        
        if self.guidance_scale < 0:
            raise ValueError("guidance_scale must be non-negative")

# Preset configurations for different use cases
class ConfigPresets:
    """Predefined configuration presets for different scenarios"""
    
    @staticmethod
    def quick_test() -> TrainingConfig:
        """Quick test configuration for debugging"""
        return TrainingConfig(
            num_epochs=5,
            batch_size=1,
            save_steps=50,
            validation_steps=25,
            lora_rank=8,
            learning_rate=5e-5
        )
    
    @staticmethod
    def production_training() -> TrainingConfig:
        """Production training configuration"""
        return TrainingConfig(
            num_epochs=200,
            batch_size=2,
            gradient_accumulation_steps=8,
            save_steps=1000,
            validation_steps=500,
            lora_rank=8,
            lora_alpha=16,
            learning_rate=5e-5,
            use_8bit_adam=True,
            gradient_checkpointing=True
        )
    
    @staticmethod
    def memory_efficient() -> TrainingConfig:
        """Memory-efficient configuration for limited GPU memory"""
        return TrainingConfig(
            batch_size=1,
            gradient_accumulation_steps=16,
            resolution=256,
            lora_rank=8,
            gradient_checkpointing=True,
            enable_cpu_offload=True,
            mixed_precision="fp16"
        )
    
    @staticmethod
    def high_quality() -> TrainingConfig:
        """High-quality training configuration"""
        return TrainingConfig(
            resolution=768,
            num_epochs=300,
            lora_rank=8,
            lora_alpha=16,
            learning_rate=5e-5,
            batch_size=1,
            gradient_accumulation_steps=32,
            use_ema=True
        )

# Utility functions
def load_config_from_args(args) -> TrainingConfig:
    """Create TrainingConfig from argparse arguments"""
    config_dict = {}
    
    # Map argument names to config attributes
    arg_mapping = {
        'data_dir': 'data_dir',
        'output_dir': 'output_dir',
        'model_name': 'model_name',
        'resolution': 'resolution',
        'batch_size': 'batch_size',
        'num_epochs': 'num_epochs',
        'learning_rate': 'learning_rate',
        'lora_rank': 'lora_rank',
        'lora_alpha': 'lora_alpha',
        'gradient_accumulation_steps': 'gradient_accumulation_steps',
        'save_steps': 'save_steps',
        'validation_steps': 'validation_steps',
        'mixed_precision': 'mixed_precision',
        'use_8bit_adam': 'use_8bit_adam'
    }
    
    for arg_name, config_key in arg_mapping.items():
        if hasattr(args, arg_name):
            value = getattr(args, arg_name)
            if value is not None:
                config_dict[config_key] = value
    
    return TrainingConfig(**config_dict)

def save_config_template(output_path: str = "./config_template.json"):
    """Save a template configuration file"""
    config = TrainingConfig()
    config.to_json(output_path)
    print(f"Configuration template saved to: {output_path}")

if __name__ == "__main__":
    # Example usage and testing
    print("Testing configuration classes...")
    
    # Test default configuration
    config = TrainingConfig()
    print(f"Default config created: {config.model_name}")
    
    # Test preset configurations
    quick_config = ConfigPresets.quick_test()
    print(f"Quick test config: {quick_config.num_epochs} epochs")
    
    prod_config = ConfigPresets.production_training()
    print(f"Production config: {prod_config.lora_rank} rank")
    
    # Test configuration saving/loading
    config.to_json("test_config.json")
    loaded_config = TrainingConfig.from_json("test_config.json")
    print(f"Config loaded successfully: {loaded_config.learning_rate}")
    
    # Test dataset configuration
    dataset_config = DatasetConfig()
    print(f"Dataset config: {len(dataset_config.supported_characters)} supported characters")
    
    print("All configuration tests passed!")

In [ ]:
%%writefile utils.py
#!/usr/bin/env python3
"""
Utility functions for brush lettering LoRA training
"""

import os
import torch
import logging
import json
import numpy as np
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont
import cv2
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

def setup_logging(output_dir: str, log_level: str = "INFO") -> None:
    """
    Setup logging configuration
    
    Args:
        output_dir: Directory to save log files
        log_level: Logging level (DEBUG, INFO, WARNING, ERROR)
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    log_file = output_path / "training.log"
    
    # Configure logging
    logging.basicConfig(
        level=getattr(logging, log_level.upper()),
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    
    logger = logging.getLogger(__name__)
    logger.info(f"Logging setup complete. Log file: {log_file}")

def save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    step: int,
    loss: float,
    config: Dict[str, Any],
    checkpoint_dir: str,
    is_best: bool = False
) -> str:
    """
    Save training checkpoint
    
    Args:
        model: Model to save
        optimizer: Optimizer state
        epoch: Current epoch
        step: Current step
        loss: Current loss
        config: Training configuration
        checkpoint_dir: Directory to save checkpoint
        is_best: Whether this is the best checkpoint
    
    Returns:
        Path to saved checkpoint
    """
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    
    # Prepare checkpoint data
    checkpoint_data = {
        'epoch': epoch,
        'step': step,
        'loss': loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'config': config
    }
    
    # Save checkpoint
    if is_best:
        checkpoint_file = checkpoint_path / "best_checkpoint.pt"
    else:
        checkpoint_file = checkpoint_path / f"checkpoint_epoch_{epoch}_step_{step}.pt"
    
    torch.save(checkpoint_data, checkpoint_file)
    
    # Save latest checkpoint link
    latest_file = checkpoint_path / "latest_checkpoint.pt"
    torch.save(checkpoint_data, latest_file)
    
    logging.info(f"Checkpoint saved: {checkpoint_file}")
    return str(checkpoint_file)

def load_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    checkpoint_path: str,
    device: torch.device = None
) -> Dict[str, Any]:
    """
    Load training checkpoint
    
    Args:
        model: Model to load weights into
        optimizer: Optimizer to load state into
        checkpoint_path: Path to checkpoint file
        device: Device to load checkpoint on
    
    Returns:
        Dictionary with checkpoint information
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    logging.info(f"Checkpoint loaded from: {checkpoint_path}")
    logging.info(f"Resuming from epoch {checkpoint['epoch']}, step {checkpoint['step']}")
    
    return {
        'epoch': checkpoint['epoch'],
        'step': checkpoint['step'],
        'loss': checkpoint['loss'],
        'config': checkpoint.get('config', {})
    }

def compute_snr(timesteps, noise_scheduler):
    """
    Compute signal-to-noise ratio for timesteps
    Used for loss weighting in diffusion training
    
    Args:
        timesteps: Tensor of timesteps
        noise_scheduler: Diffusion noise scheduler
    
    Returns:
        SNR values for the timesteps
    """
    alphas_cumprod = noise_scheduler.alphas_cumprod
    sqrt_alphas_cumprod = alphas_cumprod**0.5
    sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod) ** 0.5
    
    # Expand the tensors to match timesteps shape
    sqrt_alphas_cumprod = sqrt_alphas_cumprod[timesteps].float()
    while len(sqrt_alphas_cumprod.shape) < len(timesteps.shape):
        sqrt_alphas_cumprod = sqrt_alphas_cumprod[..., None]
    
    sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod[timesteps].float()
    while len(sqrt_one_minus_alphas_cumprod.shape) < len(timesteps.shape):
        sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod[..., None]
    
    # Compute SNR
    snr = (sqrt_alphas_cumprod / sqrt_one_minus_alphas_cumprod) ** 2
    return snr

def create_training_summary(
    losses: List[float],
    config: Dict[str, Any],
    output_dir: str,
    character_distribution: Optional[Dict[str, int]] = None
) -> None:
    """
    Create training summary with plots and statistics
    
    Args:
        losses: List of training losses
        config: Training configuration
        output_dir: Directory to save summary
        character_distribution: Distribution of characters in dataset
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Training Summary', fontsize=16, fontweight='bold')
    
    # Plot 1: Training Loss
    axes[0, 0].plot(losses, linewidth=2, color='blue', alpha=0.7)
    axes[0, 0].set_title('Training Loss Over Time')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Loss smoothed (moving average)
    if len(losses) > 10:
        window_size = max(1, len(losses) // 20)
        smoothed_losses = []
        for i in range(len(losses)):
            start_idx = max(0, i - window_size)
            end_idx = min(len(losses), i + window_size + 1)
            smoothed_losses.append(np.mean(losses[start_idx:end_idx]))
        
        axes[0, 1].plot(smoothed_losses, linewidth=2, color='red', alpha=0.7)
        axes[0, 1].set_title(f'Smoothed Training Loss (window={window_size})')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Smoothed Loss')
        axes[0, 1].grid(True, alpha=0.3)
    else:
        axes[0, 1].text(0.5, 0.5, 'Not enough data\nfor smoothing', 
                       ha='center', va='center', transform=axes[0, 1].transAxes)
        axes[0, 1].set_title('Smoothed Training Loss')
    
    # Plot 3: Character Distribution
    if character_distribution:
        chars = list(character_distribution.keys())
        counts = list(character_distribution.values())
        
        axes[1, 0].bar(chars, counts, color='green', alpha=0.7)
        axes[1, 0].set_title('Character Distribution in Dataset')
        axes[1, 0].set_xlabel('Characters')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].tick_params(axis='x', rotation=45)
    else:
        axes[1, 0].text(0.5, 0.5, 'Character distribution\nnot available', 
                       ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Character Distribution')
    
    # Plot 4: Training Configuration
    config_text = "Training Configuration:\n\n"
    key_configs = [
        'learning_rate', 'batch_size', 'num_epochs', 
        'lora_rank', 'lora_alpha', 'resolution'
    ]
    
    for key in key_configs:
        if key in config:
            config_text += f"{key}: {config[key]}\n"
    
    axes[1, 1].text(0.1, 0.9, config_text, transform=axes[1, 1].transAxes,
                   fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 1].set_title('Configuration')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    
    # Save plots
    summary_path = output_path / "training_summary.png"
    plt.savefig(summary_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save numerical summary
    summary_stats = {
        'final_loss': losses[-1] if losses else None,
        'min_loss': min(losses) if losses else None,
        'max_loss': max(losses) if losses else None,
        'avg_loss': np.mean(losses) if losses else None,
        'total_epochs': len(losses),
        'character_distribution': character_distribution,
        'config': config
    }
    
    with open(output_path / "training_summary.json", 'w') as f:
        json.dump(summary_stats, f, indent=2)
    
    logging.info(f"Training summary saved to {output_path}")

def visualize_samples(
    dataset,
    num_samples: int = 8,
    output_dir: str = "./sample_visualizations",
    title: str = "Dataset Samples"
) -> None:
    """
    Visualize samples from the dataset
    
    Args:
        dataset: Dataset to sample from
        num_samples: Number of samples to visualize
        output_dir: Directory to save visualizations
        title: Title for the visualization
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Calculate grid size
    cols = min(4, num_samples)
    rows = (num_samples + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1 or cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for i in range(num_samples):
        if i >= len(dataset):
            break
            
        sample = dataset[i]
        image = sample['images']
        character = sample['characters']
        prompt = sample['prompts']
        
        # Convert tensor to numpy for visualization
        if isinstance(image, torch.Tensor):
            # Denormalize from [-1, 1] to [0, 1]
            image_np = (image.permute(1, 2, 0).numpy() + 1) / 2
            image_np = np.clip(image_np, 0, 1)
        else:
            image_np = np.array(image)
        
        # Display image
        ax = axes[i] if num_samples > 1 else axes[0]
        ax.imshow(image_np, cmap='gray' if len(image_np.shape) == 2 else None)
        ax.set_title(f"'{character}'", fontsize=12, fontweight='bold')
        ax.axis('off')
        
        # Add prompt as text below image
        wrapped_prompt = '\n'.join([prompt[j:j+30] for j in range(0, len(prompt), 30)])
        ax.text(0.5, -0.1, wrapped_prompt, transform=ax.transAxes,
               ha='center', va='top', fontsize=8, wrap=True)
    
    # Hide remaining axes
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    
    # Save visualization
    viz_path = output_path / f"{title.lower().replace(' ', '_')}.png"
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    logging.info(f"Sample visualization saved to {viz_path}")

def preprocess_character_images(
    input_dir: str,
    output_dir: str,
    target_size: Tuple[int, int] = (512, 512),
    background_color: str = "white"
) -> Dict[str, List[str]]:
    """
    Preprocess character images for training
    
    Args:
        input_dir: Directory containing raw character images
        output_dir: Directory to save processed images
        target_size: Target size for processed images
        background_color: Background color for processed images
    
    Returns:
        Dictionary mapping characters to list of processed image paths
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    processed_images = {}
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    
    for img_file in input_path.rglob("*"):
        if img_file.suffix.lower() in image_extensions:
            try:
                # Load image
                img = Image.open(img_file)
                
                # Convert to RGB
                if img.mode != 'RGB':
                    if img.mode == 'RGBA':
                        # Handle transparency
                        background = Image.new('RGB', img.size, background_color)
                        background.paste(img, mask=img.split()[-1])
                        img = background
                    else:
                        img = img.convert('RGB')
                
                # Resize while maintaining aspect ratio
                img.thumbnail(target_size, Image.Resampling.LANCZOS)
                
                # Create new image with target size and paste centered
                new_img = Image.new('RGB', target_size, background_color)
                paste_x = (target_size[0] - img.width) // 2
                paste_y = (target_size[1] - img.height) // 2
                new_img.paste(img, (paste_x, paste_y))
                
                # Extract character from filename
                char = extract_character_from_filename(img_file.name)
                
                if char:
                    if char not in processed_images:
                        processed_images[char] = []
                    
                    # Save processed image
                    output_file = output_path / f"{char}_{len(processed_images[char]):03d}.png"
                    new_img.save(output_file, 'PNG')
                    processed_images[char].append(str(output_file))
                
            except Exception as e:
                logging.warning(f"Failed to process {img_file}: {e}")
    
    # Log statistics
    total_images = sum(len(images) for images in processed_images.values())
    logging.info(f"Processed {total_images} images for {len(processed_images)} characters")
    
    return processed_images

def extract_character_from_filename(filename: str) -> Optional[str]:
    """
    Extract character from filename
    
    Args:
        filename: Image filename
    
    Returns:
        Extracted character or None
    """
    # Remove extension
    name = Path(filename).stem.lower()
    
    # Method 1: Single character filename
    if len(name) == 1 and (name.isalpha() or name in '.,!?;:-\'"'):
        return name
    
    # Method 2: Character_number format
    if '_' in name:
        parts = name.split('_')
        if len(parts) >= 1:
            char = parts[0]
            if len(char) == 1 and (char.isalpha() or char in '.,!?;:-\'"'):
                return char
    
    # Method 3: Character followed by numbers
    import re
    match = re.match(r'^([a-z.,!?;:\-\'"]{1})\d*', name)
    if match:
        return match.group(1)
    
    return None

def calculate_dataset_statistics(dataset) -> Dict[str, Any]:
    """
    Calculate comprehensive dataset statistics
    
    Args:
        dataset: Dataset to analyze
    
    Returns:
        Dictionary with dataset statistics
    """
    if len(dataset) == 0:
        return {"error": "Empty dataset"}
    
    # Character distribution
    char_dist = dataset.get_character_distribution() if hasattr(dataset, 'get_character_distribution') else {}
    
    # Sample a few items to get image statistics
    sample_images = []
    sample_prompts = []
    
    sample_size = min(100, len(dataset))  # Sample up to 100 items
    indices = np.random.choice(len(dataset), sample_size, replace=False)
    
    for idx in indices:
        try:
            sample = dataset[idx]
            sample_images.append(sample['images'])
            sample_prompts.append(sample['prompts'])
        except:
            continue
    
    # Image statistics
    if sample_images:
        if isinstance(sample_images[0], torch.Tensor):
            image_shapes = [img.shape for img in sample_images]
            image_means = [img.mean().item() for img in sample_images]
            image_stds = [img.std().item() for img in sample_images]
        else:
            image_shapes = [np.array(img).shape for img in sample_images]
            image_means = [np.array(img).mean() for img in sample_images]
            image_stds = [np.array(img).std() for img in sample_images]
    else:
        image_shapes, image_means, image_stds = [], [], []
    
    # Prompt statistics
    prompt_lengths = [len(prompt) for prompt in sample_prompts]
    
    stats = {
        "total_samples": len(dataset),
        "character_distribution": char_dist,
        "unique_characters": len(char_dist),
        "image_statistics": {
            "shapes": list(set(map(str, image_shapes))),
            "mean_pixel_value": {
                "mean": np.mean(image_means) if image_means else 0,
                "std": np.std(image_means) if image_means else 0
            },
            "pixel_std": {
                "mean": np.mean(image_stds) if image_stds else 0,
                "std": np.std(image_stds) if image_stds else 0
            }
        },
        "prompt_statistics": {
            "length": {
                "mean": np.mean(prompt_lengths) if prompt_lengths else 0,
                "std": np.std(prompt_lengths) if prompt_lengths else 0,
                "min": min(prompt_lengths) if prompt_lengths else 0,
                "max": max(prompt_lengths) if prompt_lengths else 0
            }
        }
    }
    
    return stats

def memory_cleanup():
    """Clean up GPU and system memory"""
    import gc
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Force garbage collection
    gc.collect()

def estimate_training_time(
    dataset_size: int,
    batch_size: int,
    num_epochs: int,
    gradient_accumulation_steps: int = 1,
    seconds_per_step: float = 2.0
) -> Dict[str, str]:
    """
    Estimate training time
    
    Args:
        dataset_size: Number of samples in dataset
        batch_size: Training batch size
        num_epochs: Number of epochs
        gradient_accumulation_steps: Gradient accumulation steps
        seconds_per_step: Estimated seconds per training step
    
    Returns:
        Dictionary with time estimates
    """
    steps_per_epoch = dataset_size // (batch_size * gradient_accumulation_steps)
    total_steps = steps_per_epoch * num_epochs
    total_seconds = total_steps * seconds_per_step
    
    # Convert to human readable format
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    
    return {
        "total_steps": total_steps,
        "steps_per_epoch": steps_per_epoch,
        "estimated_time": f"{hours}h {minutes}m {seconds}s",
        "estimated_seconds": total_seconds
    }

def create_inference_pipeline(
    model_path: str,
    base_model: str = "runwayml/stable-diffusion-v1-5"
):
    """
    Create inference pipeline from trained LoRA model
    
    Args:
        model_path: Path to trained LoRA model
        base_model: Base Stable Diffusion model
    
    Returns:
        Configured pipeline for inference
    """
    from diffusers import StableDiffusionPipeline
    
    # Load base pipeline
    pipeline = StableDiffusionPipeline.from_pretrained(
        base_model,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False
    )
    
    # Load LoRA weights
    pipeline.unet.load_attn_procs(model_path)
    
    # Move to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipeline = pipeline.to(device)
    
    # Enable memory efficient attention
    if hasattr(pipeline, "enable_attention_slicing"):
        pipeline.enable_attention_slicing()
    
    if hasattr(pipeline, "enable_model_cpu_offload"):
        pipeline.enable_model_cpu_offload()
    
    return pipeline

def generate_character_samples(
    pipeline,
    characters: List[str],
    output_dir: str,
    num_samples_per_char: int = 4,
    guidance_scale: float = 12,
    num_inference_steps: int = 60
) -> Dict[str, List[str]]:
    """
    Generate sample characters using trained model
    
    Args:
        pipeline: Trained diffusion pipeline
        characters: List of characters to generate
        output_dir: Directory to save generated samples
        num_samples_per_char: Number of samples per character
        guidance_scale: Guidance scale for generation
        num_inference_steps: Number of inference steps
    
    Returns:
        Dictionary mapping characters to generated image paths
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    generated_samples = {}
    
    for char in characters:
        char_samples = []
        
        # Generate prompt
        prompt = f"brush lettering character '{char}', elegant calligraphy style, black ink on white paper"
        
        for i in range(num_samples_per_char):
            try:
                # Generate image
                with torch.autocast("cuda"):
                    image = pipeline(
                        prompt,
                        guidance_scale=guidance_scale,
                        num_inference_steps=num_inference_steps,
                        height=512,
                        width=512
                    ).images[0]
                
                # Save image
                output_file = output_path / f"generated_{char}_{i:03d}.png"
                image.save(output_file)
                char_samples.append(str(output_file))
                
            except Exception as e:
                logging.warning(f"Failed to generate sample for '{char}': {e}")
        
        generated_samples[char] = char_samples
        logging.info(f"Generated {len(char_samples)} samples for character '{char}'")
    
    return generated_samples

In [ ]:
%%writefile dataset.py
#!/usr/bin/env python3

"""
Dataset module for calligraphy character images
"""
import os
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageOps
import torchvision.transforms as transforms
import json
import random
from pathlib import Path
import numpy as np

class BrushLetteringDataset(Dataset):
    def __init__(self, data_dir, resolution=512, split='train', augment=True):
        """
        Dataset for brush lettering character images
        
        Expected directory structure:
        data_dir/
        ├── A.png
        ├── a_01.png
        ├── d.png
        ├── e.png
        └── ...
        """
        self.data_dir = Path(data_dir)
        self.resolution = resolution
        self.split = split
        self.augment = augment
        
        # Load image paths and metadata
        self.image_paths = []
        self.prompts = []
        self.characters = []
        
        self._load_data()
        self._setup_transforms()
        
        print(f"Loaded {len(self.image_paths)} images for {split} split")
    
    def _load_data(self):
        """Load image paths and generate prompts"""
        images_dir = self.data_dir 
        
        if not images_dir.exists():
            raise ValueError(f"Images directory not found: {images_dir}")
        
        # Get all image files
        image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
        all_images = []
        
        for ext in image_extensions:
            all_images.extend(list(images_dir.glob(f"*{ext}")))
            all_images.extend(list(images_dir.glob(f"*{ext.upper()}")))
        
        if not all_images:
            raise ValueError(f"No images found in {images_dir}")
        
        # Sort for consistent ordering
        all_images.sort()
        
        print(f"Found {len(all_images)} total images")
        print(f"Sample filenames: {[img.name for img in all_images[:5]]}")
        
        # Extract character information from filenames
        for img_path in all_images:
            stem = img_path.stem
            character = self._extract_character_from_filename(stem)
            
            if character:
                self.image_paths.append(img_path)
                self.characters.append(character)
                
                # Generate prompt for this character
                prompt = self._generate_character_prompt(character)
                self.prompts.append(prompt)
        
        print(f"Successfully processed {len(self.image_paths)} images")
        if self.characters:
            unique_chars = list(set(self.characters))
            print(f"Found characters: {sorted(unique_chars)}")
        
        # Split data
        if self.split in ['train', 'val']:
            total_size = len(self.image_paths)
            if total_size == 0:
                print("Warning: No valid images found!")
                return
                
            val_size = max(1, int(total_size * 0.1))  # 10% for validation
            
            if self.split == 'train':
                self.image_paths = self.image_paths[val_size:]
                self.prompts = self.prompts[val_size:]
                self.characters = self.characters[val_size:]
            else:  # validation
                self.image_paths = self.image_paths[:val_size]
                self.prompts = self.prompts[:val_size]
                self.characters = self.characters[:val_size]
        
        # Load custom prompts if available
        prompts_file = self.data_dir / "prompts.json"
        if prompts_file.exists():
            self._load_custom_prompts(prompts_file)
    
    def _extract_character_from_filename(self, filename_stem):
        """Extract character from filename stem"""
        # Handle different patterns:
        # 1. Single character: "A", "a", "d", "e" -> return as is
        # 2. Character with number: "a_01", "B_02" -> return first part
        # 3. Character with variations: "a01", "B02" -> return first character
        
        # Pattern 1: Character with underscore and number (e.g., "a_01")
        if '_' in filename_stem:
            parts = filename_stem.split('_')
            character = parts[0]
            # Validate it's a single character (letter, digit, or punctuation)
            if len(character) == 1 and (character.isalnum() or character in '.,!?;:-()[]{}'):
                return character
        
        # Pattern 2: Single character filename (e.g., "A", "d")
        elif len(filename_stem) == 1 and (filename_stem.isalnum() or filename_stem in '.,!?;:-()[]{}'):
            return filename_stem
        
        # Pattern 3: Character followed by digits (e.g., "a01", "B02")
        elif len(filename_stem) > 1:
            first_char = filename_stem[0]
            remaining = filename_stem[1:]
            # Check if first char is valid and rest are digits
            if (first_char.isalnum() or first_char in '.,!?;:-()[]{}') and remaining.isdigit():
                return first_char
        
        # If none of the patterns match, return None (skip this file)
        print(f"Warning: Could not extract character from filename: {filename_stem}")
        return None
    
    def _generate_character_prompt(self, character):
        """Generate training prompt for a character"""
        # Determine if it's uppercase, lowercase, digit, or punctuation
        if character.isupper():
            char_type = "uppercase letter"
        elif character.islower():
            char_type = "lowercase letter"
        elif character.isdigit():
            char_type = "number"
        else:
            char_type = "character"
        
        base_templates = [
            f"brush lettering {char_type} '{character}', calligraphy style",
            f"handwritten {char_type} '{character}' in brush calligraphy",
            f"elegant brush stroke {char_type} '{character}'",
            f"calligraphic {char_type} '{character}' with brush pen",
            f"artistic brush lettering '{character}'",
            f"flowing brush calligraphy {char_type} '{character}'",
            f"hand lettered '{character}' in brush style",
            f"brush script {char_type} '{character}'"
        ]
        
        style_modifiers = [
            ", black ink on white paper",
            ", elegant and flowing",
            ", artistic calligraphy",
            ", traditional brush style",
            ", modern brush lettering",
            ", expressive strokes",
            ", clean and minimal",
            ", bold brush strokes"
        ]
        
        quality_modifiers = [
            ", high quality",
            ", professional",
            ", detailed",
            ", sharp and clear",
            ", well composed",
            ", artistic",
            "",
            ""
        ]
        
        base = random.choice(base_templates)
        style = random.choice(style_modifiers)
        quality = random.choice(quality_modifiers)
        
        return base + style + quality
    
    def _load_custom_prompts(self, prompts_file):
        """Load custom prompts from JSON file"""
        try:
            with open(prompts_file, 'r') as f:
                custom_prompts = json.load(f)
            
            # Update prompts if custom ones are provided
            for i, char in enumerate(self.characters):
                if char in custom_prompts:
                    if isinstance(custom_prompts[char], list):
                        self.prompts[i] = random.choice(custom_prompts[char])
                    else:
                        self.prompts[i] = custom_prompts[char]
        except Exception as e:
            print(f"Warning: Could not load custom prompts: {e}")
    
    def _setup_transforms(self):
        """Setup image transformations"""
        transforms_list = []
        
        # Base transforms
        transforms_list.extend([
            transforms.Resize((self.resolution, self.resolution), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
        
        # Augmentation transforms for training
        if self.augment and self.split == 'train':
            augment_transforms = [
                transforms.RandomRotation(degrees=5, fill=1.0),
                transforms.RandomPerspective(distortion_scale=0.1, p=0.3, fill=1.0),
                transforms.RandomAffine(
                    degrees=3,
                    translate=(0.05, 0.05),
                    scale=(0.95, 1.05),
                    shear=2,
                    fill=1.0
                ),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.RandomErasing(p=0.1, scale=(0.02, 0.05), ratio=(0.3, 3.3), value=1.0),
            ]
            
            # Add augmentations randomly
            for aug in augment_transforms:
                if random.random() < 0.3:
                    transforms_list.insert(-1, aug)
        
        # Normalization (to [-1, 1] range expected by SD)
        transforms_list.append(transforms.Normalize([0.5], [0.5]))
        
        self.transform = transforms.Compose(transforms_list)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path)
        
        # Convert to RGB if needed (SD expects RGB)
        if image.mode != 'RGB':
            if image.mode == 'L':
                image = ImageOps.colorize(image, black="black", white="white")
            else:
                image = image.convert('RGB')
        
        # Apply transforms
        image = self.transform(image)
        
        # Get prompt
        prompt = self.prompts[idx]
        character = self.characters[idx]
        
        return {
            'images': image,
            'prompts': prompt,
            'characters': character,
            'image_paths': str(img_path)
        }
    
    def get_character_distribution(self):
        """Get distribution of characters in dataset"""
        from collections import Counter
        return Counter(self.characters)
    
    def save_sample_batch(self, save_dir, num_samples=5):
        """Save sample images and prompts for inspection"""
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        
        samples = []
        for i in range(min(num_samples, len(self))):
            sample = self[i]
            
            # Convert tensor back to PIL image
            image_tensor = sample['images']
            # Denormalize
            image_tensor = (image_tensor + 1) / 2
            image_tensor = torch.clamp(image_tensor, 0, 1)
            
            # Convert to PIL
            image_pil = transforms.ToPILImage()(image_tensor)
            
            # Save image
            image_path = save_dir / f"sample_{i}_{sample['characters']}.png"
            image_pil.save(image_path)
            
            samples.append({
                'character': sample['characters'],
                'prompt': sample['prompts'],
                'image_path': str(image_path),
                'original_path': sample['image_paths']
            })
        
        # Save metadata
        with open(save_dir / "samples_metadata.json", 'w') as f:
            json.dump(samples, f, indent=2)
        
        print(f"Saved {len(samples)} sample images to {save_dir}")


# Utility function to analyze your dataset
def analyze_dataset_files(data_dir):
    """Analyze the files in your dataset directory"""
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"Directory not found: {data_path}")
        return None, []
    
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    images = []
    
    for ext in image_extensions:
        images.extend(list(data_path.glob(f"*{ext}")))
        images.extend(list(data_path.glob(f"*{ext.upper()}")))
    
    if images:
        print(f"\nFound {len(images)} images in: {data_path}")
        print("Sample filenames:")
        for img in images[:15]:  # Show more samples
            print(f"  {img.name}")
        
        # Analyze filename patterns and extract characters
        characters_found = []
        patterns = {'single_char': [], 'char_underscore_num': [], 'char_num': [], 'other': []}
        
        for img in images:
            stem = img.stem
            
            # Single character
            if len(stem) == 1 and (stem.isalnum() or stem in '.,!?;:-()[]{}'):
                patterns['single_char'].append(stem)
                characters_found.append(stem)
            # Character with underscore and number
            elif '_' in stem:
                parts = stem.split('_')
                if len(parts[0]) == 1 and (parts[0].isalnum() or parts[0] in '.,!?;:-()[]{}'):
                    patterns['char_underscore_num'].append(stem)
                    characters_found.append(parts[0])
                else:
                    patterns['other'].append(stem)
            # Character followed by digits
            elif len(stem) > 1 and stem[0].isalnum() and stem[1:].isdigit():
                patterns['char_num'].append(stem)
                characters_found.append(stem[0])
            else:
                patterns['other'].append(stem)
        
        print("\nFilename patterns detected:")
        for pattern, examples in patterns.items():
            if examples:
                unique_examples = list(set(examples))[:5]
                print(f"  {pattern}: {len(examples)} files, examples: {unique_examples}")
        
        print(f"\nUnique characters found: {sorted(set(characters_found))}")
        print(f"Total characters: {len(characters_found)}")
        
        return data_path, images
    else:
        print("No images found!")
        return None, []


if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", required=True, help="Path to dataset directory")
    parser.add_argument("--analyze", action="store_true", help="Analyze dataset files")
    parser.add_argument("--save_samples", action="store_true", help="Save sample images")
    args = parser.parse_args()
    
    if args.analyze:
        print("Analyzing dataset files...")
        analyze_dataset_files(args.data_dir)
    
    # Test dataset loading
    try:
        dataset = BrushLetteringDataset(args.data_dir, resolution=512, split='train')
        print(f"\nDataset loaded successfully!")
        print(f"Dataset size: {len(dataset)}")
        print(f"Character distribution: {dataset.get_character_distribution()}")
        
        # Test loading a sample
        if len(dataset) > 0:
            sample = dataset[0]
            print(f"\nSample:")
            print(f"  Character: {sample['characters']}")
            print(f"  Prompt: {sample['prompts']}")
            print(f"  Image shape: {sample['images'].shape}")
            print(f"  Original path: {sample['image_paths']}")
        
        # Save samples if requested
        if args.save_samples:
            dataset.save_sample_batch("./sample_outputs")
            
    except Exception as e:
        print(f"\nError loading dataset: {e}")
        print("\nTry running with --analyze flag first to check your file structure")

In [ ]:
%%writefile main.py
#!/usr/bin/env python3
"""
Main training script for calligraphy LoRA model 
"""

import os
import sys
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import argparse
from pathlib import Path
import logging
from tqdm import tqdm
import json
import gc

# For Jupyter/Colab - set arguments programmatically
sys.argv = [
    'main.py',
    '--data_dir', '/kaggle/input/single-character-a-images',
    '--output_dir', './outputs',
    '--num_epochs', '50',
    '--batch_size', '1',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--learning_rate', '5e-5',
    '--gradient_accumulation_steps', '4',
    '--save_steps', '500',
    '--validation_steps', '100',
    '--mixed_precision', 'fp16',
    '--use_8bit_adam',
    '--resolution', '512'
]

from dataset import BrushLetteringDataset
from utils import setup_logging, save_checkpoint, load_checkpoint, compute_snr
from config import TrainingConfig

def parse_args():
    parser = argparse.ArgumentParser(description="Train LoRA for brush lettering generation")
    parser.add_argument("--data_dir", type=str, required=True, help="Directory containing character images")
    parser.add_argument("--output_dir", type=str, default="./outputs", help="Output directory for checkpoints")
    parser.add_argument("--model_name", type=str, default="runwayml/stable-diffusion-v1-5", help="Base SD model")
    parser.add_argument("--resolution", type=int, default=512, help="Training resolution")
    parser.add_argument("--batch_size", type=int, default=1, help="Batch size")
    parser.add_argument("--num_epochs", type=int, default=30, help="Number of training epochs")
    parser.add_argument("--learning_rate", type=float, default=5e-5, help="Learning rate")
    parser.add_argument("--lora_rank", type=int, default=8, help="LoRA rank")
    parser.add_argument("--lora_alpha", type=int, default=16, help="LoRA alpha")
    parser.add_argument("--gradient_accumulation_steps", type=int, default=4, help="Gradient accumulation steps")
    parser.add_argument("--save_steps", type=int, default=500, help="Save checkpoint every N steps")
    parser.add_argument("--validation_steps", type=int, default=100, help="Run validation every N steps")
    parser.add_argument("--resume_from", type=str, default=None, help="Resume training from checkpoint")
    parser.add_argument("--mixed_precision", type=str, default="fp16", choices=["no", "fp16", "bf16"])
    parser.add_argument("--use_8bit_adam", action="store_true", help="Use 8-bit Adam optimizer")
    
    return parser.parse_args()

class LoRATrainer:
    def __init__(self, config):
        print("Initializing LoRA Trainer...")
        self.config = config
        self.setup_models()
        self.setup_optimizer()
        self.setup_scheduler()
        
    def setup_models(self):
        """Initialize and setup models with LoRA"""
        print(f"Loading base model: {self.config.model_name}")
        
        # Load the pipeline
        self.pipeline = StableDiffusionPipeline.from_pretrained(
            self.config.model_name,
            torch_dtype=torch.float16 if self.config.mixed_precision == "fp16" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False
        )
        
        # Extract components
        self.vae = self.pipeline.vae
        self.tokenizer = self.pipeline.tokenizer
        self.text_encoder = self.pipeline.text_encoder
        self.unet = self.pipeline.unet
        self.scheduler = self.pipeline.scheduler
        
        # Freeze base models
        self.vae.requires_grad_(False)
        self.text_encoder.requires_grad_(False)
        self.unet.requires_grad_(False)
        
        # Setup LoRA for UNet
        lora_config = LoraConfig(
            r=self.config.lora_rank,
            lora_alpha=self.config.lora_alpha,
            target_modules=[
                "to_k", "to_q", "to_v", "to_out.0",
                "proj_in", "proj_out",
                "ff.net.0.proj", "ff.net.2"
            ],
            lora_dropout=0.1,
        )
        
        self.unet = get_peft_model(self.unet, lora_config)
        
        # Move to device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.vae.to(device)
        self.text_encoder.to(device)
        self.unet.to(device)
        
        print(f"Models loaded and moved to {device}")
        
        # Print trainable parameters
        trainable_params = sum(p.numel() for p in self.unet.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.unet.parameters())
        print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
    
    def setup_optimizer(self):
        """Setup optimizer"""
        if self.config.use_8bit_adam:
            try:
                import bitsandbytes as bnb
                optimizer_cls = bnb.optim.AdamW8bit
            except ImportError:
                raise ImportError("Please install bitsandbytes: pip install bitsandbytes")
        else:
            optimizer_cls = torch.optim.AdamW
        
        self.optimizer = optimizer_cls(
            self.unet.parameters(),
            lr=self.config.learning_rate,
            betas=(0.9, 0.999),
            weight_decay=1e-2,
            eps=1e-08,
        )
    
    def setup_scheduler(self):
        """Setup noise scheduler"""
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            self.config.model_name, 
            subfolder="scheduler"
        )
    
    def encode_prompt(self, prompt):
        """Encode text prompt"""
        text_inputs = self.tokenizer(
            prompt,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        
        with torch.no_grad():
            text_embeddings = self.text_encoder(
                text_inputs.input_ids.to(self.text_encoder.device)
            )[0]
        
        return text_embeddings
    
    def training_step(self, batch):
        """Single training step"""
        images = batch["images"].to(self.unet.device)
        prompts = batch["prompts"]
        
        # FIX: Ensure images match VAE dtype
        if self.config.mixed_precision == "fp16":
            images = images.to(torch.float16)
        
        # Encode images to latent space
        with torch.no_grad():
            latents = self.vae.encode(images).latent_dist.sample()
            latents = latents * self.vae.config.scaling_factor
        
        # Sample noise
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        
        # Sample random timesteps
        timesteps = torch.randint(
            0, self.noise_scheduler.config.num_train_timesteps, 
            (bsz,), device=latents.device
        ).long()
        
        # Add noise to latents
        noisy_latents = self.noise_scheduler.add_noise(latents, noise, timesteps)
        
        # Encode prompts
        encoder_hidden_states = self.encode_prompt(prompts)
        
        # Predict noise
        model_pred = self.unet(
            noisy_latents, 
            timesteps, 
            encoder_hidden_states=encoder_hidden_states
        ).sample
        
        # Compute loss
        if self.noise_scheduler.config.prediction_type == "epsilon":
            target = noise
        elif self.noise_scheduler.config.prediction_type == "v_prediction":
            target = self.noise_scheduler.get_velocity(latents, noise, timesteps)
        else:
            raise ValueError(f"Unknown prediction type {self.noise_scheduler.config.prediction_type}")
        
        loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
        
        return loss
    
    def validation_step(self, val_dataloader):
        """Validation step"""
        self.unet.eval()
        total_loss = 0
        num_batches = 0
        
        with torch.no_grad():
            for batch in val_dataloader:
                loss = self.training_step(batch)
                total_loss += loss.item()
                num_batches += 1
                
                if num_batches >= 10:  # Limit validation batches
                    break
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0
        self.unet.train()
        return avg_loss
    
    def train(self, train_dataloader, val_dataloader=None):
        """Main training loop"""
        print("Starting training...")
        
        global_step = 0
        epoch_losses = []
        
        # Setup mixed precision
        scaler = torch.cuda.amp.GradScaler() if self.config.mixed_precision == "fp16" else None
        
        for epoch in range(self.config.num_epochs):
            print(f"\nEpoch {epoch + 1}/{self.config.num_epochs}")
            
            epoch_loss = 0
            progress_bar = tqdm(train_dataloader, desc=f"Training Epoch {epoch + 1}")
            
            for step, batch in enumerate(progress_bar):
                with torch.cuda.amp.autocast(enabled=self.config.mixed_precision == "fp16"):
                    loss = self.training_step(batch)
                    loss = loss / self.config.gradient_accumulation_steps
                
                # Backward pass
                if scaler is not None:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()
                
                epoch_loss += loss.item()
                
                # Update weights
                if (step + 1) % self.config.gradient_accumulation_steps == 0:
                    if scaler is not None:
                        scaler.step(self.optimizer)
                        scaler.update()
                    else:
                        self.optimizer.step()
                    
                    self.optimizer.zero_grad()
                    global_step += 1
                    
                    # Update progress bar
                    progress_bar.set_postfix({
                        'loss': f'{loss.item() * self.config.gradient_accumulation_steps:.4f}',
                        'step': global_step
                    })
                    
                    # Validation
                    if val_dataloader and global_step % self.config.validation_steps == 0:
                        val_loss = self.validation_step(val_dataloader)
                        print(f"\nValidation loss: {val_loss:.4f}")
                    
                    # Save checkpoint
                    if global_step % self.config.save_steps == 0:
                        self.save_checkpoint(global_step, epoch, loss.item())
                
                # Clean up memory
                if step % 50 == 0:
                    torch.cuda.empty_cache()
                    gc.collect()
            
            avg_epoch_loss = epoch_loss / len(train_dataloader)
            epoch_losses.append(avg_epoch_loss)
            print(f"Epoch {epoch + 1} average loss: {avg_epoch_loss:.4f}")
            
            # Save epoch checkpoint
            self.save_checkpoint(global_step, epoch, avg_epoch_loss, is_epoch_end=True)
        
        print("Training completed!")
        return epoch_losses
    
    def save_checkpoint(self, step, epoch, loss, is_epoch_end=False):
        """Save training checkpoint"""
        checkpoint_dir = Path(self.config.output_dir) / "checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Save LoRA weights
        if is_epoch_end:
            lora_path = checkpoint_dir / f"lora_epoch_{epoch}"
        else:
            lora_path = checkpoint_dir / f"lora_step_{step}"
        
        self.unet.save_pretrained(lora_path)
        
        # Save training state
        state_dict = {
            'step': step,
            'epoch': epoch,
            'loss': loss,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'config': self.config.__dict__
        }
        
        state_path = lora_path.parent / f"training_state_{step}.pt"
        torch.save(state_dict, state_path)
        
        print(f"Checkpoint saved: {lora_path}")

def main():
    args = parse_args()
    
    # Setup logging
    setup_logging(args.output_dir)
    
    # Create config
    config = TrainingConfig(
        data_dir=args.data_dir,
        output_dir=args.output_dir,
        model_name=args.model_name,
        resolution=args.resolution,
        batch_size=args.batch_size,
        num_epochs=args.num_epochs,
        learning_rate=args.learning_rate,
        lora_rank=args.lora_rank,
        lora_alpha=args.lora_alpha,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        save_steps=args.save_steps,
        validation_steps=args.validation_steps,
        mixed_precision=args.mixed_precision,
        use_8bit_adam=args.use_8bit_adam
    )
    
    # Create datasets
    print("Creating datasets...")
    train_dataset = BrushLetteringDataset(
        data_dir=config.data_dir,
        resolution=config.resolution,
        split='train'
    )
    
    val_dataset = BrushLetteringDataset(
        data_dir=config.data_dir,
        resolution=config.resolution,
        split='val'
    )
    
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    ) if len(val_dataset) > 0 else None
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset) if val_dataset else 0}")
    
    # Create trainer and start training
    trainer = LoRATrainer(config)
    
    # Resume from checkpoint if specified
    if args.resume_from:
        load_checkpoint(trainer, args.resume_from)
    
    # Train the model
    losses = trainer.train(train_dataloader, val_dataloader)
    
    # Save final model
    final_path = Path(config.output_dir) / "final_model"
    trainer.unet.save_pretrained(final_path)
    print(f"Final model saved to: {final_path}")

if __name__ == "__main__":
    main()

In [ ]:
!pip install bitsandbytes

In [ ]:
import sys
sys.argv = ['main.py',
    '--data_dir', '/kaggle/input/single-character-a-images',
    '--output_dir', './outputs',
    '--num_epochs', '50',
    '--batch_size', '1',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--learning_rate', '5e-5',
    '--gradient_accumulation_steps', '4',
    '--save_steps', '500',
    '--validation_steps', '100',
    '--mixed_precision', 'fp16',
    '--use_8bit_adam',
    '--resolution', '512']
exec(open('main.py').read())

In [ ]:
%%writefile calligraphy_generator.py
#!/usr/bin/env python3
"""
Calligraphy Generation Script using trained LoRA model
"""

import torch
import pandas as pd
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from peft import PeftModel
import argparse
from pathlib import Path
from PIL import Image, ImageOps, ImageEnhance
import numpy as np
import cv2
import json
from typing import List, Dict, Tuple
import re

class EnhancedCalligraphyGenerator:
    def __init__(self, pipe):
        self.pipe = pipe

    def generate_calligraphy(self, char: str, steps=50, guidance=8.0):
        prompt = f"brush lettering character '{char}', elegant calligraphy style, black ink on white paper"
        result = self.pipe(prompt=prompt, num_inference_steps=steps, guidance_scale=guidance)
        return result.images[0]


class CalligraphyGenerator:
    def __init__(self, model_path: str, lora_path: str, device: str = "auto"):
        """
        Initialize the calligraphy generator
        """
        self.device = self._setup_device(device)
        self.model_path = model_path
        self.lora_path = lora_path

        print(f"Loading models on {self.device}...")
        self._load_models()

    def _setup_device(self, device: str) -> str:
        if device == "auto":
            return "cuda" if torch.cuda.is_available() else "cpu"
        return device

    def _load_models(self):
        self.pipe = StableDiffusionPipeline.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False
        )

        if Path(self.lora_path).exists():
            print(f"Loading LoRA weights from: {self.lora_path}")
            self.pipe.unet = PeftModel.from_pretrained(
                self.pipe.unet, 
                self.lora_path,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
            )
        else:
            raise FileNotFoundError(f"LoRA weights not found at: {self.lora_path}")

        self.pipe = self.pipe.to(self.device)
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(self.pipe.scheduler.config)
 
        if hasattr(self.pipe, "enable_xformers_memory_efficient_attention"):
            try:
                self.pipe.enable_xformers_memory_efficient_attention()
            except Exception:
                pass

        self.pipe.enable_attention_slicing()
        print("Models loaded successfully!")
        self.enhancedGenerator = EnhancedCalligraphyGenerator(self.pipe)
        
    def _create_calligraphy_prompt(self, text: str, style_emphasis: str = "medium") -> str:
        import random
        is_single_char = len(text) == 1

        if is_single_char:
            char_templates = [
                f"brush lettering character '{text}', elegant calligraphy style",
                f"handwritten letter '{text}' in brush calligraphy",
                f"brush lettering character '{text}', artistic style",
                f"elegant {self._get_char_type(text)} '{text}' with {self._get_stroke_desc(text)}",
                f"flowing {self._get_char_type(text)} '{text}' in brush calligraphy"
            ]
            base_templates = char_templates
        else:
            word_templates = [
                f"brush lettering word '{text}' in flowing script",
                f"elegant brush calligraphy word '{text}'",
                f"handwritten word '{text}' in brush style",
                f"artistic brush lettering '{text}'",
                f"flowing brush script word '{text}'"
            ]
            base_templates = word_templates

        style_modifiers = {
            "light": [
                ", clean black ink on white paper",
                ", simple elegant strokes",
                ", minimalist calligraphy style"
            ],
            "medium": [
                ", black ink on white background",
                ", expressive brush strokes",
                ", flowing calligraphy with confident strokes",
                ", artistic hand lettering style"
            ],
            "strong": [
                ", bold expressive calligraphy, dramatic strokes",
                ", confident brush lettering, artistic style",
                ", dynamic calligraphy with flowing movement",
                ", professional brush calligraphy, black ink on white"
            ]
        }

        quality_terms = [
            ", high contrast, clean lines",
            ", professional calligraphy, sharp details",
            ", clear brush strokes, artistic quality",
            ", elegant penmanship, high resolution"
        ]

        negative_elements = [
            "blurry", "low quality", "pixelated", "distorted", "multiple copies",
            "duplicate text", "cropped", "worst quality", "jpeg artifacts",
            "watermark", "signature", "username", "logo", "copyright",
            "multiple characters", "repeated letters"
        ]

        base = random.choice(base_templates)
        style = random.choice(style_modifiers.get(style_emphasis, style_modifiers["medium"]))
        quality = random.choice(quality_terms)

        prompt = base + style + quality
        negative_prompt = ", ".join(negative_elements)
        logger.info("\nprompt : ",prompt,"\n")
        return prompt, negative_prompt

    def _get_char_type(self, char: str) -> str:
        if char.islower():
            return "lowercase"
        elif char.isupper():
            return "uppercase"
        elif char.isdigit():
            return "numeral"
        else:
            return "character"

    def _get_stroke_desc(self, char: str) -> str:
        stroke_descriptions = {
            'a': 'elegant strokes', 'b': 'tall ascender', 'c': 'curved form',
            'd': 'curved bowl', 'e': 'flowing curves', 'f': 'elegant ascender',
            'g': 'descender loop', 'h': 'tall ascender', 'i': 'simple dot',
            'j': 'curved descender', 'k': 'angled strokes', 'l': 'tall ascender',
            'm': 'double arch', 'n': 'smooth arch', 'o': 'circular form',
            'p': 'descender', 'q': 'curved tail', 'r': 'smooth shoulder',
            's': 'curved form', 't': 'crossbar', 'u': 'curved bottom',
            'v': 'angled strokes', 'w': 'wide form', 'x': 'crossed strokes',
            'y': 'descender', 'z': 'zigzag',
            'A': 'triangular form', 'B': 'double bowls', 'C': 'curved form',
            'D': 'curved bowl', 'E': 'horizontal bars', 'F': 'horizontal bars',
            'G': 'curved form', 'H': 'crossbar', 'I': 'vertical stem',
            'J': 'curved hook', 'K': 'angled strokes', 'L': 'horizontal base',
            'M': 'majestic form', 'N': 'diagonal stroke', 'O': 'circular form',
            'P': 'closed bowl', 'Q': 'curved tail', 'R': 'bowl and leg',
            'S': 'curved form', 'T': 'horizontal top', 'U': 'curved form',
            'V': 'angled strokes', 'W': 'wide form', 'X': 'crossed strokes',
            'Y': 'graceful form', 'Z': 'zigzag',
            '0': 'circular form', '1': 'clean stem', '2': 'curved form',
            '3': 'double curves', '4': 'angular form', '5': 'curved bottom',
            '6': 'curved form', '7': 'angled stroke', '8': 'curved form',
            '9': 'curved top'
        }
        return stroke_descriptions.get(char, 'artistic strokes')

    def _post_process_image(self, image: Image.Image, target_dpi: int = 1200) -> Image.Image:
        img_array = np.array(image)
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(1.3)
        enhancer = ImageEnhance.Sharpness(image)
        image = enhancer.enhance(1.2)

        if image.mode != 'RGB':
            image = image.convert('RGB')

        img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        lab = cv2.cvtColor(img_cv, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img_cv = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        image = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
        image.info['dpi'] = (target_dpi, target_dpi)
        return image

    def generate_single_text(self, text: str, style_emphasis: str = "medium",
                             num_inference_steps: int = 60, guidance_scale: float = 12,
                             height: int = 768, width: int = 1024, seed: int = None) -> Image.Image:
        prompt, negative_prompt = self._create_calligraphy_prompt(text, style_emphasis)
        print(f"Generating calligraphy for: '{text}'")
        print(f"Prompt: {prompt}")

        if seed is not None:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

        with torch.autocast("cuda" if self.device == "cuda" else "cpu"):
            result = self.pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                height=height,
                width=width,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                num_images_per_prompt=1
            )
        image = result.images[0]
        return self._post_process_image(image)

    def generate_from_csv(self, csv_path: str, output_dir: str,
                          text_column: str = "text", name_column: str = "name",
                          **generation_kwargs) -> List[Dict]:
        df = pd.read_csv(csv_path)
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        results = []

        for idx, row in df.iterrows():
            text = str(row[text_column])
            if name_column in df.columns and pd.notna(row[name_column]):
                filename = f"{idx:03d}_{self._clean_filename(str(row[name_column]))}.png"
            else:
                filename = f"{idx:03d}_{self._clean_filename(text)}.png"
            filepath = output_path / filename

            try:
                image = self.generate_single_text(text, **generation_kwargs)
                image.save(filepath, "PNG", dpi=(1200, 1200))
                result = {
                    "index": idx,
                    "text": text,
                    "filename": filename,
                    "filepath": str(filepath),
                    "status": "success"
                }
                print(f"Generated: {filename}")
            except Exception as e:
                result = {
                    "index": idx,
                    "text": text,
                    "filename": filename,
                    "filepath": str(filepath),
                    "status": "error",
                    "error": str(e)
                }
                print(f"Error generating {filename}: {e}")

            results.append(result)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        results_path = output_path / "generation_results.json"
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)

        print(f"\nGeneration complete! Results saved to: {output_path}")
        print(f"Summary: {sum(1 for r in results if r['status'] == 'success')}/{len(results)} successful")
        return results

    def _clean_filename(self, text: str) -> str:
        text = re.sub(r'[<>:"/\\|?*]', '_', text)
        text = re.sub(r'\s+', '_', text)
        return text[:50]

    def generate_batch_samples(self, texts: List[str], output_dir: str,
                               variations_per_text: int = 3,
                               **generation_kwargs) -> List[Dict]:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        results = []

        for text_idx, text in enumerate(texts):
            for var_idx in range(variations_per_text):
                filename = f"text_{text_idx:03d}_var_{var_idx:02d}_{self._clean_filename(text)}.png"
                filepath = output_path / filename

                try:
                    seed = text_idx * 1000 + var_idx if 'seed' not in generation_kwargs else None
                    image = self.generate_single_text(text, seed=seed, **generation_kwargs)
                    image.save(filepath, "PNG", dpi=(1200, 1200))
                    result = {
                        "text_index": text_idx,
                        "variation": var_idx,
                        "text": text,
                        "filename": filename,
                        "filepath": str(filepath),
                        "status": "success"
                    }
                    print(f"Generated: {filename}")
                except Exception as e:
                    result = {
                        "text_index": text_idx,
                        "variation": var_idx,
                        "text": text,
                        "filename": filename,
                        "filepath": str(filepath),
                        "status": "error",
                        "error": str(e)
                    }
                    print(f"Error generating {filename}: {e}")

                results.append(result)

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        return results

def main():
    parser = argparse.ArgumentParser(description="Generate calligraphy using trained LoRA model")
    parser.add_argument("--lora_path", required=True, help="Path to trained LoRA weights")
    parser.add_argument("--model_path", default="runwayml/stable-diffusion-v1-5", help="Base model path")
    parser.add_argument("--output_dir", default="./generated_calligraphy", help="Output directory")
    parser.add_argument("--text", type=str, help="Single text to generate")
    parser.add_argument("--csv_path", type=str, help="Path to CSV file with texts")
    parser.add_argument("--text_column", default="text", help="Column name for text in CSV")
    parser.add_argument("--name_column", default="name", help="Column name for names in CSV")
    parser.add_argument("--style_emphasis", default="medium", choices=["light", "medium", "strong"])
    parser.add_argument("--num_inference_steps", type=int, default=50)
    parser.add_argument("--guidance_scale", type=float, default=12)
    parser.add_argument("--height", type=int, default=768)
    parser.add_argument("--width", type=int, default=1024)
    parser.add_argument("--seed", type=int, help="Random seed for reproducibility")
    parser.add_argument("--variations", type=int, default=1, help="Number of variations per text")

    args = parser.parse_args()

    generator = CalligraphyGenerator(
        model_path=args.model_path,
        lora_path=args.lora_path
    )

    gen_kwargs = {
        "style_emphasis": args.style_emphasis,
        "num_inference_steps": args.num_inference_steps,
        "guidance_scale": args.guidance_scale,
        "height": args.height,
        "width": args.width,
        "seed": args.seed
    }

    if args.text:
        print(f"Generating calligraphy for: '{args.text}'")
        image = generator.enhancedGenerator.generate_calligraphy(args.text)
        output_path = Path(args.output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        filename = f"{generator._clean_filename(args.text)}.png"
        filepath = output_path / filename
        image.save(filepath, "PNG", dpi=(1200, 1200))
        print(f"Saved: {filepath}")
    elif args.csv_path:
        print(f"Generating calligraphy from CSV: {args.csv_path}")
        results = generator.generate_from_csv(
            csv_path=args.csv_path,
            output_dir=args.output_dir,
            text_column=args.text_column,
            name_column=args.name_column,
            **gen_kwargs
        )
        print(f"Generated {len(results)} images")
    else:
        demo_texts = [
            "Hello Beautiful",
            "Wedding Invitation",
            "Thank You",
            "Save the Date",
            "Congratulations"
        ]
        print("Demo mode - generating sample texts...")
        results = generator.generate_batch_samples(
            texts=demo_texts,
            output_dir=args.output_dir,
            variations_per_text=args.variations,
            **gen_kwargs
        )
        print(f"Generated {len(results)} sample images")

if __name__ == "__main__":
    main()


In [ ]:
def generate_brush_lettering_prompt(text: str, style_variant: str = "elegant") -> str:
    """
    Generate properly formatted prompts for brush lettering model
    
    Args:
        text: The character(s) or word to generate
        style_variant: Style descriptor ("elegant", "flowing", "artistic", "dynamic")
    
    Returns:
        Formatted prompt string
    """
    
    # Style variations
    style_templates = {
        "elegant": {
            "single_char": "brush lettering character '{text}', elegant calligraphy style",
            "word": "elegant brush calligraphy word '{text}'"
        },
        "flowing": {
            "single_char": "flowing {char_type} '{text}' in brush calligraphy", 
            "word": "brush lettering word '{text}' in flowing script"
        },
        "artistic": {
            "single_char": "brush lettering character '{text}', artistic style",
            "word": "artistic brush lettering '{text}'"
        },
        "dynamic": {
            "single_char": "dynamic {char_type} '{text}' with {stroke_desc}",
            "word": "handwritten word '{text}' in brush style"
        },
        "handwritten": {
            "single_char": "handwritten letter '{text}' in brush calligraphy",
            "word": "handwritten word '{text}' in brush calligraphy"
        }
    }
    
    # Determine if single character or word
    is_single_char = len(text) == 1
    
    if is_single_char:
        # Determine character type and stroke description for dynamic style
        char_type = get_character_type(text)
        stroke_desc = get_stroke_description(text)
        
        template = style_templates[style_variant]["single_char"]
        
        if style_variant == "dynamic":
            return template.format(text=text, char_type=char_type, stroke_desc=stroke_desc)
        elif style_variant == "flowing":
            return template.format(text=text, char_type=char_type)
        else:
            return template.format(text=text)
    else:
        # Multiple characters - treat as word
        template = style_templates[style_variant]["word"]
        return template.format(text=text)

def get_character_type(char: str) -> str:
    """Determine character type for prompt formatting"""
    if char.islower():
        return "lowercase"
    elif char.isupper():
        return "uppercase"
    elif char.isdigit():
        return "numeral"
    else:
        return "character"

def get_stroke_description(char: str) -> str:
    """Get appropriate stroke description for character"""
    stroke_descriptions = {
        # Lowercase with descenders
        'g': 'descender loop', 'j': 'curved descender', 'p': 'descender', 
        'q': 'curved tail', 'y': 'descender',
        
        # Lowercase with ascenders  
        'b': 'tall ascender', 'd': 'curved bowl', 'f': 'flowing ascender',
        'h': 'tall ascender', 'k': 'angled strokes', 'l': 'tall ascender',
        't': 'crossbar',
        
        # Special lowercase
        'i': 'dot', 'm': 'double arch', 'n': 'smooth arch', 'o': 'circular form',
        'r': 'smooth shoulder', 's': 'curved form', 'u': 'curved bottom',
        'v': 'angled strokes', 'w': 'wide form', 'x': 'crossed strokes', 'z': 'zigzag',
        
        # Uppercase
        'A': 'triangular form', 'B': 'double bowls', 'C': 'curved form', 'D': 'curved bowl',
        'E': 'horizontal bars', 'F': 'horizontal bars', 'G': 'curved form', 'H': 'crossbar',
        'I': 'vertical stem', 'J': 'curved hook', 'K': 'angled strokes', 'L': 'horizontal base',
        'M': 'majestic form', 'N': 'diagonal stroke', 'O': 'circular form', 'P': 'closed bowl',
        'Q': 'tail', 'R': 'bowl and leg', 'S': 'curved form', 'T': 'horizontal top',
        'U': 'curved bottom', 'V': 'angled strokes', 'W': 'wide form', 'X': 'crossed strokes',
        'Y': 'graceful form', 'Z': 'zigzag',
        
        # Numbers
        '0': 'circular form', '1': 'clean stem', '2': 'curved form', '3': 'double curves',
        '4': 'angular form', '5': 'curved bottom', '6': 'curved form', '7': 'angled stroke',
        '8': 'double curves', '9': 'curved top'
    }
    
    return stroke_descriptions.get(char, 'artistic strokes')

# Example usage functions
def generate_single_character_prompts(char: str) -> list:
    """Generate multiple prompt variations for a single character"""
    styles = ["elegant", "flowing", "artistic", "dynamic", "handwritten"]
    return [generate_brush_lettering_prompt(char, style) for style in styles]

def generate_word_prompts(word: str) -> list:
    """Generate multiple prompt variations for a word"""
    styles = ["elegant", "flowing", "artistic", "handwritten"]
    return [generate_brush_lettering_prompt(word, style) for style in styles]

# Test the functions
if __name__ == "__main__":
    # Test single character
    print("Single character 'd' prompts:")
    for prompt in generate_single_character_prompts('d'):
        print(f"  - {prompt}")
    
    print("\nWord 'dog' prompts:")
    for prompt in generate_word_prompts('dog'):
        print(f"  - {prompt}")
    
    # Test various characters
    test_chars = ['A', 'g', '3', '!']
    print(f"\nTesting various characters:")
    for char in test_chars:
        prompt = generate_brush_lettering_prompt(char, "elegant")
        print(f"  {char}: {prompt}")

In [ ]:
!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "brush lettering character 'a', elegant calligraphy style, sharp details, high resolution" \
    --output_dir "./generated"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "flowing script lowercase 'a', graceful curves, smooth lines, dark ink" \
    --output_dir "./generated"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "expressive brush stroke 'a', artistic calligraphy, dynamic lines, clean background" \
    --output_dir "./generated"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "crisp brush written 'a', delicate details, high quality, professional" \
    --output_dir "./generated"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "beautiful calligraphic 'a' with flourishes, classic style" \
    --output_dir "./generated"

In [ ]:
!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "bold brush lettering uppercase 'A', strong calligraphic style, clean definition, high resolution" \
    --output_dir "./generated/A"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "formal capital 'A' in elegant calligraphy, refined strokes, precise" \
    --output_dir "./generated/A"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "majestic brush stroke 'A', artistic calligraphy, commanding presence, dark ink" \
    --output_dir "./generated/A"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "flowing script capital 'A', graceful curves, intricate details, high quality" \
    --output_dir "./generated/A"

!python calligraphy_generator.py \
    --lora_path "./outputs/final_model" \
    --text "ornate calligraphic uppercase 'A', elaborate flourishes, traditional style" \
    --output_dir "./generated/A"